In [1]:
!pip install "textdistance[extras]"
!pip install cerebras-cloud-sdk


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 5.6 MB/s eta 0:00:00


In [2]:
!pip install -q -U trl transformers peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 36.1 MB/s eta 0:00:00


In [3]:
import os
try:
    from google.colab import userdata

    # Load the key from Colab Secrets
    api_key = userdata.get('CEREBRAS_API_KEY')
    os.environ["CEREBRAS_API_KEY"] = api_key
    api_key_deepseek = userdata.get('DEEPSEEK_API_KEY')
    os.environ["DEEPSEEK_API_KEY"] = api_key_deepseek

    print("✅ API Keys loaded successfully from Colab Secrets.")

except ImportError:
    print("❌ Not running in Colab or 'userdata' not found.")
except Exception as e:
    print(f"❌ Error loading key: {e}")

✅ API Keys loaded successfully from Colab Secrets.


In [4]:
import os
import json
import re
from cerebras.cloud.sdk import Cerebras

client = Cerebras(
    api_key=os.environ.get("CEREBRAS_API_KEY")
)

JUDGE_MODEL_ID = "zai-glm-4.6"

JUDGE_SYSTEM_PROMPT = """
You are a senior SQL database administrator acting as a strict code reviewer.
LANGUAGE CONSTRAINT: You MUST always respond in English.
FORMAT CONSTRAINT: You MUST output a single valid JSON object. Do not output markdown blocks or conversational text.

Your task is to evaluate a Candidate SQL query against a Gold Standard SQL query.

Evaluation Rules:
1. LOGIC CHECK: Determine if the Candidate SQL returns the EXACT same data as the Gold SQL.
2. OPTIMIZATION CHECK: If logic is correct, determine if the Candidate SQL is efficient (avoids redundant subqueries, unnecessary distincts, etc).

Output JSON Schema:
{
  "logic_correct": boolean,
  "is_optimized": boolean,
  "reasoning": "string (concise explanation)"
}
"""

JUDGE_USER_TEMPLATE = """
Schema: {schema}
User Request: {question}

Gold SQL: {gold_sql}
Candidate SQL: {gen_sql}
"""

def extract_part(text, tag):
    """Helper to extract Schema/Question from the training prompt"""
    match = re.search(f"{tag}: (.*?)(?=\\n|$)", text, re.DOTALL)
    return match.group(1).strip() if match else "Unknown"

def llm_judge_reward(prompts, completions, answer, **kwargs):
    rewards = []

    for prompt, gen_sql, gold_sql in zip(prompts, completions, answer):
        try:
            schema_text = extract_part(prompt, "Schema")
            question_text = extract_part(prompt, "User Request")

            response = client.chat.completions.create(
                model=JUDGE_MODEL_ID,
                messages=[
                    {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                    {"role": "user", "content": JUDGE_USER_TEMPLATE.format(
                        schema=schema_text,
                        question=question_text,
                        gold_sql=gold_sql,
                        gen_sql=gen_sql
                    )}
                ],
                temperature=0.6,
                top_p=0.95,
                max_completion_tokens=1024,
                response_format={"type": "json_object"},
                stream=False
            )

            content = response.choices[0].message.content


            clean_json = content.replace("```json", "").replace("```", "").strip()
            result = json.loads(clean_json)

            if not result.get('logic_correct', False):
                rewards.append(0.0)
            else:
                if result.get('is_optimized', False):
                    rewards.append(1.5)
                else:
                    rewards.append(1.0)

        except Exception as e:
            print(f"Cerebras Judge Error: {e}")
            rewards.append(0.0)

    return rewards

In [10]:
import os
import json
import re
from openai import OpenAI

# Initialize OpenAI client compatible with DeepSeek API
client = OpenAI(
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

# DeepSeek-V3 is accessed via 'deepseek-chat'
JUDGE_MODEL_ID = "deepseek-chat"

JUDGE_SYSTEM_PROMPT = """
You are a senior SQL database administrator acting as a strict code reviewer.
LANGUAGE CONSTRAINT: You MUST always respond in English.
FORMAT CONSTRAINT: You MUST output a single valid JSON object. Do not output markdown blocks or conversational text.

Your task is to evaluate a Candidate SQL query against a Gold Standard SQL query.

Evaluation Rules:
1. LOGIC CHECK: Determine if the Candidate SQL returns the EXACT same data as the Gold SQL.
2. OPTIMIZATION CHECK: If logic is correct, determine if the Candidate SQL is efficient (avoids redundant subqueries, unnecessary distincts, etc).

Output JSON Schema:
{
  "logic_correct": boolean,
  "is_optimized": boolean,
  "reasoning": "string (concise explanation)"
}
"""

JUDGE_USER_TEMPLATE = """
Schema: {schema}
User Request: {question}

Gold SQL: {gold_sql}
Candidate SQL: {gen_sql}
"""

def extract_part(text, tag):
    """Helper to extract Schema/Question from the training prompt"""
    match = re.search(f"{tag}: (.*?)(?=\\n|$)", text, re.DOTALL)
    return match.group(1).strip() if match else "Unknown"

def llm_judge_reward_deepseek(prompts, completions, answer, **kwargs):
    rewards = []

    for prompt, gen_sql, gold_sql in zip(prompts, completions, answer):
        try:
            schema_text = extract_part(prompt, "Schema")
            question_text = extract_part(prompt, "User Request")

            # DeepSeek supports the standard OpenAI chat completions format
            response = client.chat.completions.create(
                model=JUDGE_MODEL_ID,
                messages=[
                    {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                    {"role": "user", "content": JUDGE_USER_TEMPLATE.format(
                        schema=schema_text,
                        question=question_text,
                        gold_sql=gold_sql,
                        gen_sql=gen_sql
                    )}
                ],
                temperature=0.6,
                top_p=0.95,
                max_tokens=1024, # Note: DeepSeek uses max_tokens, not max_completion_tokens
                response_format={"type": "json_object"},
                stream=False
            )

            content = response.choices[0].message.content

            # Sanitizing just in case, though json_object mode is usually reliable
            clean_json = content.replace("```json", "").replace("```", "").strip()
            result = json.loads(clean_json)

            if not result.get('logic_correct', False):
                rewards.append(0.0)
            else:
                if result.get('is_optimized', False):
                    rewards.append(1.5)
                else:
                    rewards.append(1.0)

        except Exception as e:
            print(f"DeepSeek Judge Error: {e}")
            rewards.append(0.0)

    return rewards

In [11]:
test_prompts = [
    # Sample 1
    """<|im_start|>system
    You are a SQL expert.
    Schema: CREATE TABLE users (id INT, name TEXT, age INT);
    <|im_end|>
    <|im_start|>user
    User Request: Get all user names who are 25.
    <|im_end|>""",

    # Sample 2
    """<|im_start|>system
    Schema: CREATE TABLE orders (id INT, amount INT);
    <|im_end|>
    <|im_start|>user
    User Request: Total revenue?
    <|im_end|>""",

    # Sample 3
    """<|im_start|>system
    Schema: CREATE TABLE items (id INT);
    <|im_end|>
    <|im_start|>user
    User Request: Count items.
    <|im_end|>"""
]

# Sample Generated Candidates
test_completions = [
    "SELECT name FROM users WHERE age = 25",             # Good Logic, Optimized
    "SELECT * FROM users WHERE age = 25",                # Good Logic, Bad Optimization (Select *)
    "SELECT name FROM users WHERE age > 25"              # Wrong Logic (Wrong symbol)
]

# Sample Gold Answers
test_answers = [
    "SELECT name FROM users WHERE age = 25",
    "SELECT * FROM users WHERE age = 25",
    "SELECT count(*) FROM items"
]

scores = llm_judge_reward_deepseek(test_prompts, test_completions, test_answers)

scores

[1.5, 1.5, 0.0]

In [7]:
import sqlite3
import sqlglot

def sql_syntax_reward(completions, **kwargs):
    rewards = []
    for completion in completions:
        try:
            sqlglot.transpile(sql_query)
            rewards.append(0.0)
        except:
            rewards.append(-2.0)
    return rewards

def execution_reward(prompts, completions, answer, **kwargs):
    """
    Executes generated SQL and Ground Truth SQL on a temporary DB.
    'answer' comes from your dataset (the gold standard SQL).
    """
    rewards = []
    conn = sqlite3.connect(":memory:")
    cursor = conn.cursor()

    for completion, gold_sql in zip(completions, answer):
        try:
            gen_sql = completion.split("<sql>")[1].split("</sql>")[0].strip()

            cursor.execute(gen_sql)
            gen_result = cursor.fetchall()

            cursor.execute(gold_sql)
            gold_result = cursor.fetchall()

            if set(gen_result) == set(gold_result):
                rewards.append(2.0)
            else:
                rewards.append(0.0)

        except Exception:
            rewards.append(-1.0)

    conn.close()
    return rewards

In [8]:
import pandas as pd
from datasets import Dataset, DatasetDict

deepseek_path = '/content/deepseek_generation.csv'
gemini_path = '/content/gemini_generation.csv'
qwen_path = '/content/qwen_7b_generation.csv'
schema_path = '/content/gdelt_schema.txt'

with open(schema_path, 'r') as f:
    gdelt_schema = f.read().strip()

df1 = pd.read_csv(deepseek_path)
df2 = pd.read_csv(gemini_path)
df3 = pd.read_csv(qwen_path)
full_df = pd.concat([df1, df2, df3], ignore_index=True)

def format_grpo_prompt(row):
    system_message = (
        "You are a SQL expert. Generate a SQL query for the GDELT 2.0 BigQuery dataset.\n"
        f"Here is the database schema:\n{gdelt_schema}\n\n"
        "Strictly output only the SQL query inside <sql> tags."
    )
    user_message = row['text_query']

    return (
        f"<|im_start|>system\n{system_message}<|im_end|>\n"
        f"<|im_start|>user\n{user_message}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

full_df['prompt'] = full_df.apply(format_grpo_prompt, axis=1)
full_df['answer'] = full_df['sql_query'].apply(
    lambda x: x if "<sql>" in x else f"<sql>\n{x}\n</sql>"
)

full_dataset = Dataset.from_pandas(full_df[['prompt', 'answer']])

train_split = full_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = train_split['train']
test_val_dataset = train_split['test']

val_split = test_val_dataset.train_test_split(test_size=0.5, seed=42)
val_dataset = val_split['train']
test_dataset = val_split['test']

dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

print(f"Train Size: {len(dataset['train'])} ({len(dataset['train'])/len(full_dataset):.0%})")
print(f"Val Size:   {len(dataset['validation'])} ({len(dataset['validation'])/len(full_dataset):.0%})")
print(f"Test Size:  {len(dataset['test'])} ({len(dataset['test'])/len(full_dataset):.0%})")

Train Size: 1432 (80%)
Val Size:   179 (10%)
Test Size:  179 (10%)


In [ ]:
import os
import torch
from trl import GRPOTrainer, GRPOConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading Model in BFloat16 (A100 Native)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
)

model.config.torch_dtype = torch.bfloat16
model.config.use_cache = False

model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
    bias="none"
)
model = get_peft_model(model, peft_config)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

training_args = GRPOConfig(
    output_dir="grpo_sql_model",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_generations=4,
    max_completion_length=512,
    beta=0.1,
    fp16=False,
    bf16=True,
    logging_steps=10,
    report_to="none"
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[sql_syntax_reward, execution_reward, llm_judge_reward_deepseek],
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    peft_config=None,
    processing_class=tokenizer,
)

print("Starting training on A100...")
trainer.train()

Loading Model in BFloat16 (A100 Native)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting training on A100...


Step,Training Loss
